# DeepFake Detection — Dataset Exploration

This notebook explores the deepfake detection datasets:
- Real vs. fake sample counts per dataset
- Sample visualizations
- FFT spectrum comparison (real vs. fake)
- Frequency-domain artifact analysis

In [ ]:
import sys
sys.path.insert(0, '..')  # Add project root

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
from pathlib import Path
from PIL import Image
import cv2

# Project imports
from src.datasets import (
    FaceForensicsPlusPlus,
    CelebDFv2,
    DF40Dataset,
    WildDeepfake,
    get_val_transforms,
)
from src.models.frequency_branch import compute_fft_spectrum, compute_dct, rgb_to_grayscale

print('Imports successful!')

## 1. Dataset Configuration

In [ ]:
import yaml

with open('../configs/default.yaml', 'r') as f:
    config = yaml.safe_load(f)

DATA_ROOT = config['datasets']['data_root']
print(f'Data root: {DATA_ROOT}')
print(f'Datasets configured: {list(config["datasets"].keys())}')

## 2. Dataset Statistics

In [ ]:
transform = get_val_transforms(img_size=224)
ds_cfg = config['datasets']

datasets = {}
stats = {}

# Try to load each dataset
dataset_configs = [
    ('FF++', FaceForensicsPlusPlus, {
        'root': ds_cfg.get('ff_plus_plus', {}).get('root', 'data/ff++'),
        'split': 'train',
        'transform': transform,
    }),
    ('Celeb-DF v2', CelebDFv2, {
        'root': ds_cfg.get('celeb_df', {}).get('root', 'data/celeb_df'),
        'split': 'train',
        'transform': transform,
    }),
    ('DF40', DF40Dataset, {
        'root': ds_cfg.get('df40', {}).get('root', 'data/df40'),
        'split': 'train',
        'transform': transform,
    }),
    ('WildDeepfake', WildDeepfake, {
        'root': ds_cfg.get('wild_deepfake', {}).get('root', 'data/wild_deepfake'),
        'split': 'train',
        'transform': transform,
    }),
]

for name, DatasetClass, kwargs in dataset_configs:
    try:
        ds = DatasetClass(**kwargs)
        labels = ds.get_labels()
        n_real = labels.count(0)
        n_fake = labels.count(1)
        total = len(labels)
        stats[name] = {
            'total': total,
            'real': n_real,
            'fake': n_fake,
            'fake_ratio': n_fake / total if total > 0 else 0,
        }
        datasets[name] = ds
        print(f'{name}: {total} total | {n_real} real | {n_fake} fake')
    except Exception as e:
        print(f'{name}: Not available ({e})')
        stats[name] = {'total': 0, 'real': 0, 'fake': 0, 'fake_ratio': 0}

In [ ]:
# Dataset statistics bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = list(stats.keys())
totals = [stats[n]['total'] for n in names]
reals = [stats[n]['real'] for n in names]
fakes = [stats[n]['fake'] for n in names]

x = np.arange(len(names))
width = 0.35

# Grouped bar chart
ax1 = axes[0]
bars1 = ax1.bar(x - width/2, reals, width, label='Real', color='#4CAF50', alpha=0.85, edgecolor='white')
bars2 = ax1.bar(x + width/2, fakes, width, label='Fake', color='#F44336', alpha=0.85, edgecolor='white')
ax1.set_xlabel('Dataset', fontsize=12)
ax1.set_ylabel('Sample Count', fontsize=12)
ax1.set_title('Real vs. Fake Sample Counts', fontsize=13, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(names, rotation=15, ha='right')
ax1.legend(fontsize=11)
ax1.grid(True, axis='y', alpha=0.3)

# Add count labels
for bar in bars1:
    h = bar.get_height()
    if h > 0:
        ax1.text(bar.get_x() + bar.get_width()/2, h + max(totals)*0.01,
                f'{int(h):,}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    h = bar.get_height()
    if h > 0:
        ax1.text(bar.get_x() + bar.get_width()/2, h + max(totals)*0.01,
                f'{int(h):,}', ha='center', va='bottom', fontsize=8)

# Fake ratio pie/bar
ax2 = axes[1]
fake_ratios = [stats[n]['fake_ratio'] for n in names]
colors_ratio = ['#F44336' if r >= 0.5 else '#4CAF50' for r in fake_ratios]
bars = ax2.barh(names, fake_ratios, color=colors_ratio, alpha=0.8, edgecolor='white')
ax2.axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='50% balance')
ax2.set_xlabel('Fake Ratio', fontsize=12)
ax2.set_title('Fake Sample Ratio per Dataset', fontsize=13, fontweight='bold')
ax2.set_xlim(0, 1)
ax2.legend(fontsize=11)
ax2.grid(True, axis='x', alpha=0.3)

for bar, ratio in zip(bars, fake_ratios):
    ax2.text(ratio + 0.01, bar.get_y() + bar.get_height()/2,
            f'{ratio:.2%}', va='center', fontsize=10)

plt.suptitle('Dataset Statistics', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/dataset_stats.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to outputs/dataset_stats.png')

## 3. Sample Visualization

In [ ]:
def show_samples(dataset, n_real=4, n_fake=4, title='Dataset Samples'):
    """Display sample real and fake images from a dataset."""
    from src.datasets.augmentations import denormalize
    
    real_samples = []
    fake_samples = []
    
    for i in range(min(len(dataset), 500)):
        sample = dataset[i]
        if sample['label'].item() == 0 and len(real_samples) < n_real:
            real_samples.append(sample)
        elif sample['label'].item() == 1 and len(fake_samples) < n_fake:
            fake_samples.append(sample)
        if len(real_samples) >= n_real and len(fake_samples) >= n_fake:
            break
    
    all_samples = real_samples + fake_samples
    n = len(all_samples)
    if n == 0:
        print('No samples available.')
        return
    
    n_cols = min(8, n)
    n_rows = (n + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2, n_rows * 2.5))
    axes = axes.flatten() if n_rows > 1 or n_cols > 1 else [axes]
    
    for i, (ax, sample) in enumerate(zip(axes, all_samples)):
        img = denormalize(sample['image'])
        img_np = (img.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        ax.imshow(img_np)
        label = 'FAKE' if sample['label'].item() == 1 else 'REAL'
        color = '#F44336' if label == 'FAKE' else '#4CAF50'
        ax.set_title(f'{label}\n{sample["manipulation_type"]}',
                    fontsize=7, color=color, fontweight='bold')
        ax.axis('off')
    
    for ax in axes[n:]:
        ax.axis('off')
    
    plt.suptitle(title, fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# Show samples from available datasets
for name, ds in datasets.items():
    if len(ds) > 0:
        print(f'\n{name} samples:')
        show_samples(ds, n_real=4, n_fake=4, title=f'{name} — Real vs. Fake Samples')
        break  # Show one dataset to avoid long runtime

## 4. FFT Spectrum Analysis — Real vs. Fake

In [ ]:
def compute_mean_fft_spectrum(dataset, n_samples=50, label_filter=0):
    """
    Compute mean FFT spectrum over N samples with a given label.
    label_filter: 0 = real, 1 = fake
    """
    from src.datasets.augmentations import denormalize
    
    spectra = []
    count = 0
    
    for i in range(min(len(dataset), 2000)):
        if count >= n_samples:
            break
        
        sample = dataset[i]
        if sample['label'].item() != label_filter:
            continue
        
        # Denormalize image
        img = denormalize(sample['image'])  # [3, H, W] in [0,1]
        img_batch = img.unsqueeze(0)  # [1, 3, H, W]
        
        # Compute FFT spectrum
        gray = rgb_to_grayscale(img_batch)  # [1, 1, H, W]
        spectrum = compute_fft_spectrum(gray)  # [1, 1, H, W]
        spectra.append(spectrum[0, 0].numpy())  # [H, W]
        count += 1
    
    if not spectra:
        return None
    
    return np.mean(spectra, axis=0)


def plot_fft_comparison(dataset, n_samples=50, title='FFT Spectrum: Real vs. Fake'):
    """Compare mean FFT spectra of real vs. fake images."""
    print(f'Computing FFT spectra for {n_samples} real and fake samples...')
    
    real_spectrum = compute_mean_fft_spectrum(dataset, n_samples=n_samples, label_filter=0)
    fake_spectrum = compute_mean_fft_spectrum(dataset, n_samples=n_samples, label_filter=1)
    
    if real_spectrum is None or fake_spectrum is None:
        print('Not enough samples for both classes.')
        return
    
    # Difference spectrum
    diff_spectrum = fake_spectrum - real_spectrum
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    im1 = axes[0].imshow(real_spectrum, cmap='viridis', aspect='auto')
    axes[0].set_title('Mean FFT Spectrum — REAL', fontsize=12, fontweight='bold')
    axes[0].axis('off')
    plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
    
    im2 = axes[1].imshow(fake_spectrum, cmap='viridis', aspect='auto')
    axes[1].set_title('Mean FFT Spectrum — FAKE', fontsize=12, fontweight='bold')
    axes[1].axis('off')
    plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
    
    vmax = np.abs(diff_spectrum).max()
    im3 = axes[2].imshow(diff_spectrum, cmap='RdBu_r', aspect='auto',
                         vmin=-vmax, vmax=vmax)
    axes[2].set_title('Difference (Fake - Real)', fontsize=12, fontweight='bold')
    axes[2].axis('off')
    plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04)
    
    plt.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('../outputs/fft_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('FFT comparison saved to outputs/fft_comparison.png')


# Run on first available dataset
for name, ds in datasets.items():
    if len(ds) > 10:
        plot_fft_comparison(ds, n_samples=20, title=f'FFT Spectrum Analysis — {name}')
        break

## 5. Frequency Domain Artifact Visualization (Single Sample)

In [ ]:
def visualize_frequency_artifacts(dataset, real_idx=0, fake_idx=None):
    """
    Show RGB image, FFT spectrum, and DCT coefficients side by side
    for both a real and fake sample.
    """
    from src.datasets.augmentations import denormalize
    
    # Find one real and one fake sample
    real_sample = None
    fake_sample = None
    
    for i in range(min(len(dataset), 500)):
        s = dataset[i]
        if s['label'].item() == 0 and real_sample is None:
            real_sample = s
        if s['label'].item() == 1 and fake_sample is None:
            fake_sample = s
        if real_sample is not None and fake_sample is not None:
            break
    
    samples = []
    if real_sample: samples.append(('REAL', real_sample))
    if fake_sample: samples.append(('FAKE', fake_sample))
    
    if not samples:
        print('No samples available.')
        return
    
    fig, axes = plt.subplots(len(samples), 3, figsize=(12, 4 * len(samples)))
    if len(samples) == 1:
        axes = axes[np.newaxis, :]
    
    col_titles = ['RGB Image', 'FFT Spectrum (log)', 'DCT Coefficients']
    
    for row, (label_str, sample) in enumerate(samples):
        # RGB image
        img = denormalize(sample['image'])  # [3, H, W]
        img_batch = img.unsqueeze(0)        # [1, 3, H, W]
        img_np = (img.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        
        # FFT
        gray = rgb_to_grayscale(img_batch)
        fft_spec = compute_fft_spectrum(gray)[0, 0].numpy()
        
        # DCT
        dct_coeff = compute_dct(gray)[0, 0].numpy()
        
        # Plot
        color = '#F44336' if label_str == 'FAKE' else '#4CAF50'
        
        axes[row, 0].imshow(img_np)
        axes[row, 0].set_title(f'{label_str} — {col_titles[0]}',
                               fontsize=11, color=color, fontweight='bold')
        axes[row, 0].axis('off')
        
        im1 = axes[row, 1].imshow(fft_spec, cmap='hot')
        axes[row, 1].set_title(f'{label_str} — {col_titles[1]}',
                               fontsize=11, color=color, fontweight='bold')
        axes[row, 1].axis('off')
        plt.colorbar(im1, ax=axes[row, 1], fraction=0.046, pad=0.04)
        
        im2 = axes[row, 2].imshow(dct_coeff, cmap='plasma')
        axes[row, 2].set_title(f'{label_str} — {col_titles[2]}',
                               fontsize=11, color=color, fontweight='bold')
        axes[row, 2].axis('off')
        plt.colorbar(im2, ax=axes[row, 2], fraction=0.046, pad=0.04)
    
    plt.suptitle('Frequency Domain Analysis: Real vs. Fake',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('../outputs/frequency_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Frequency analysis saved to outputs/frequency_analysis.png')


for name, ds in datasets.items():
    if len(ds) > 5:
        print(f'Frequency analysis for {name}:')
        visualize_frequency_artifacts(ds)
        break

## 6. Radially Averaged FFT Power Spectrum

In [ ]:
def radial_average(spectrum_2d):
    """Compute radially averaged power spectrum."""
    h, w = spectrum_2d.shape
    cy, cx = h // 2, w // 2
    Y, X = np.ogrid[:h, :w]
    R = np.sqrt((X - cx)**2 + (Y - cy)**2).astype(int)
    
    max_r = min(cy, cx)
    radial = np.array([
        spectrum_2d[R == r].mean() if (R == r).sum() > 0 else 0
        for r in range(max_r)
    ])
    return radial


def plot_radial_fft(dataset, n_samples=30, title='Radial FFT Power Spectrum'):
    """Compare 1D radially averaged FFT power spectra for real vs. fake."""
    from src.datasets.augmentations import denormalize
    
    real_spectra = []
    fake_spectra = []
    
    for i in range(min(len(dataset), 1000)):
        if len(real_spectra) >= n_samples and len(fake_spectra) >= n_samples:
            break
        
        s = dataset[i]
        label = s['label'].item()
        
        if (label == 0 and len(real_spectra) >= n_samples) or \
           (label == 1 and len(fake_spectra) >= n_samples):
            continue
        
        img = denormalize(s['image']).unsqueeze(0)
        gray = rgb_to_grayscale(img)
        fft = compute_fft_spectrum(gray)[0, 0].numpy()
        radial = radial_average(fft)
        
        if label == 0:
            real_spectra.append(radial)
        else:
            fake_spectra.append(radial)
    
    if not real_spectra or not fake_spectra:
        print('Not enough samples.')
        return
    
    # Align lengths
    min_len = min(min(len(r) for r in real_spectra), min(len(r) for r in fake_spectra))
    real_arr = np.array([r[:min_len] for r in real_spectra])
    fake_arr = np.array([r[:min_len] for r in fake_spectra])
    
    real_mean = real_arr.mean(0)
    real_std = real_arr.std(0)
    fake_mean = fake_arr.mean(0)
    fake_std = fake_arr.std(0)
    
    freqs = np.arange(min_len)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    
    ax.plot(freqs, real_mean, '-', color='#4CAF50', linewidth=2, label=f'Real (n={len(real_spectra)})')
    ax.fill_between(freqs, real_mean - real_std, real_mean + real_std,
                   alpha=0.2, color='#4CAF50')
    
    ax.plot(freqs, fake_mean, '-', color='#F44336', linewidth=2, label=f'Fake (n={len(fake_spectra)})')
    ax.fill_between(freqs, fake_mean - fake_std, fake_mean + fake_std,
                   alpha=0.2, color='#F44336')
    
    ax.set_xlabel('Spatial Frequency (radial bins)', fontsize=12)
    ax.set_ylabel('Mean Log-Magnitude', fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../outputs/radial_fft.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Radial FFT plot saved to outputs/radial_fft.png')


for name, ds in datasets.items():
    if len(ds) > 20:
        plot_radial_fft(ds, n_samples=20, title=f'Radial FFT Power Spectrum — {name}')
        break

## 7. Summary

In [ ]:
print('=' * 60)
print('  DATASET EXPLORATION SUMMARY')
print('=' * 60)

total_samples = sum(stats[n]['total'] for n in stats)
total_real = sum(stats[n]['real'] for n in stats)
total_fake = sum(stats[n]['fake'] for n in stats)

print(f'\nTotal samples: {total_samples:,}')
print(f'  Real: {total_real:,} ({100*total_real/max(total_samples,1):.1f}%)')
print(f'  Fake: {total_fake:,} ({100*total_fake/max(total_samples,1):.1f}%)')

print('\nPer-dataset breakdown:')
for name, s in stats.items():
    if s['total'] > 0:
        print(f'  {name:<20}: {s["total"]:>7,} total | {s["real"]:>6,} real | {s["fake"]:>6,} fake')
    else:
        print(f'  {name:<20}: NOT AVAILABLE')

print('\nModel architecture summary:')
print('  Spatial branch: DINOv2 ViT-B/14 → 768-d embedding')
print('  Frequency branch: FFT+DCT → LightCNN → 256-d embedding')
print('  Fusion MLP: 1024-d → 512 → 128 → 1 (logit)')
print('  Calibration: Temperature scaling')
print('=' * 60)